# POC — Dataset A: Riesgo por equipo (Gestión MPA)

Prueba de concepto en **Google Colab** del dataset de predicción de riesgo por equipo (Sprint 6).

| Concepto | Detalle |
|----------|--------|
| **Unidad** | 1 fila = 1 equipo |
| **Target** | `nivel_riesgo` → Bajo, Medio, Alto, Critico |
| **CSV recomendado** | `equipos_riesgo_v200.csv` (200 filas) |
| **Diccionario** | Ver `ml/data/README.md` en el repo |

## Cómo usar en Colab

1. **Archivo → Subir a Colab** (o abrir desde Drive).
2. En la celda de carga, elige **una** opción: subir CSV, montar Drive o clonar el repo.
3. Ejecuta todas las celdas en orden (`Runtime → Run all`).

> El pipeline replica la lógica de `ml/scripts/train_model.py` para comparar métricas con el modelo del proyecto (~95% accuracy en sintético).

In [ ]:
# Dependencias (Colab ya trae pandas/sklearn; instalamos solo si hace falta)
!pip -q install pandas scikit-learn matplotlib seaborn joblib

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Entorno listo.")

## 1. Cargar el dataset

Descomenta **solo una** opción en la celda siguiente.

In [ ]:
# --- Opción A (recomendada para POC rápida): subir CSV desde tu PC ---
from google.colab import files

uploaded = files.upload()  # selecciona equipos_riesgo_v200.csv
csv_path = Path(list(uploaded.keys())[0])
print(f"Archivo cargado: {csv_path} ({csv_path.stat().st_size:,} bytes)")

# --- Opción B: montar Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# csv_path = Path('/content/drive/MyDrive/gestion_mpa/ml/data/synthetic/equipos_riesgo_v200.csv')

# --- Opción C: clonar repo (ajusta la URL si usas fork) ---
# !git clone -q https://github.com/TU_USUARIO/gestion_mpa.git
# csv_path = Path('gestion_mpa/ml/data/synthetic/equipos_riesgo_v200.csv')

In [ ]:
df = pd.read_csv(csv_path)
print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")
df.head()

## 2. Exploración inicial (EDA)

In [ ]:
TARGET = "nivel_riesgo"
ID_COLS = ["equipo_id", "codigo_patrimonial"]
META_COLS = ["marca", "modelo", "area_nombre", "score_riesgo"]

print("=== Info general ===")
display(df.info())

print("\n=== Valores nulos (% por columna) ===")
null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(null_pct[null_pct > 0].round(1).to_frame("nulos_%"))

print("\n=== Distribución del target ===")
target_counts = df[TARGET].value_counts().sort_index()
display(target_counts.to_frame("conteo"))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot(kind="bar", ax=ax[0], color="steelblue", title="nivel_riesgo")
ax[0].set_xlabel("")
ax[0].tick_params(axis="x", rotation=0)

if "tipo_equipo" in df.columns:
    df.groupby("tipo_equipo")[TARGET].value_counts(normalize=True).unstack(fill_value=0).plot(
        kind="bar", stacked=True, ax=ax[1], title="Riesgo por tipo_equipo"
    )
    ax[1].set_xlabel("")
    ax[1].legend(title=TARGET, bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Features numéricas clave para correlación con score_riesgo
num_cols = [
    "antiguedad_meses", "total_mantenimientos", "correctivos_12m",
    "preventivos_12m", "severidad_max_historica", "fallas_altas_criticas_12m",
    "score_riesgo",
]
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
plt.title("Correlación — features numéricas")
plt.tight_layout()
plt.show()

## 3. Validar reglas del target

Comprobamos que `nivel_riesgo` sigue las reglas documentadas en `ml/data/README.md`.

In [ ]:
def regla_esperada(row):
    if row.get("estado_operativo") == "Dañado":
        return "Critico"
    if row.get("fallas_altas_criticas_12m", 0) >= 2:
        return "Critico"
    if row.get("fallas_altas_criticas_12m", 0) >= 1:
        return "Alto"
    if row.get("correctivos_12m", 0) >= 3:
        return "Alto"
    if row.get("antiguedad_meses", 0) > 48:
        return "Medio"
    if row.get("estado_conservacion") in ("Regular", "Malo"):
        return "Medio"
    return "Bajo"

check = df.copy()
check["riesgo_esperado"] = check.apply(regla_esperada, axis=1)
check["coincide"] = check[TARGET] == check["riesgo_esperado"]

pct_ok = check["coincide"].mean() * 100
print(f"Coincidencia con reglas documentadas: {pct_ok:.1f}%")

if pct_ok < 100:
    print("\nFilas con discrepancia (muestra):")
    display(check.loc[~check["coincide"], ["equipo_id", TARGET, "riesgo_esperado",
                                              "estado_operativo", "correctivos_12m",
                                              "fallas_altas_criticas_12m", "antiguedad_meses",
                                              "estado_conservacion"]].head(10))

## 4. Entrenamiento baseline (Random Forest)

Misma definición de features que `ml/scripts/features.py` y `train_model.py`.

In [ ]:
NUMERIC_FEATURES = [
    "ram_gb", "almacenamiento_gb", "antiguedad_meses",
    "total_mantenimientos", "correctivos_12m", "preventivos_12m",
    "dias_desde_ultimo_mantenimiento", "severidad_max_historica",
    "fallas_altas_criticas_12m", "area_id",
    "horas_uso", "errores_smart", "salud_bateria", "contador_paginas",
    "ultima_temp_cpu", "ultima_temp_disco",
]
CATEGORICAL_FEATURES = [
    "tipo_equipo", "tipo_disco", "estado_conservacion", "estado_operativo",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def preparar_dataframe(raw: pd.DataFrame) -> pd.DataFrame:
    out = raw.copy()
    for col in NUMERIC_FEATURES:
        if col not in out.columns:
            out[col] = 0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0)
    for col in CATEGORICAL_FEATURES:
        if col not in out.columns:
            out[col] = "Desconocido"
        out[col] = out[col].fillna("Desconocido").astype(str)
    return out


def build_pipeline() -> Pipeline:
    numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Desconocido")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ])
    classifier = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    return Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])


prepared = preparar_dataframe(df)
X = prepared[FEATURE_COLUMNS]
y = prepared[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = build_pipeline()
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print("\nReferencia proyecto (sintético v200): accuracy ~0.95, F1 macro ~0.92")
print("\n" + classification_report(y_test, y_pred, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap="Blues")
ax.set_title("Matriz de confusión — test set")
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features (post one-hot, nombres aproximados)
rf = pipeline.named_steps["classifier"]
pre = pipeline.named_steps["preprocessor"]

cat_names = []
if hasattr(pre.named_transformers_["cat"], "get_feature_names_out"):
    cat_names = list(pre.named_transformers_["cat"].get_feature_names_out())
feature_names = NUMERIC_FEATURES + cat_names

importances = pd.Series(rf.feature_importances_, index=feature_names[: len(rf.feature_importances_)])
top = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
top.sort_values().plot(kind="barh", color="teal")
plt.title("Top 15 importancias — Random Forest")
plt.xlabel("importancia")
plt.tight_layout()
plt.show()

## 5. Inferencia de ejemplo

Predicción sobre equipos del test set y comparación con el target real.

In [ ]:
sample_idx = X_test.index[:8]
sample = prepared.loc[sample_idx, FEATURE_COLUMNS]
preds = pipeline.predict(sample)
proba = pipeline.predict_proba(sample)
classes = pipeline.classes_

result = pd.DataFrame({
    "equipo_id": df.loc[sample_idx, "equipo_id"].values if "equipo_id" in df.columns else sample_idx,
    "real": y.loc[sample_idx].values,
    "predicho": preds,
})
result["prob_max"] = proba.max(axis=1).round(3)
result["ok"] = result["real"] == result["predicho"]
display(result)

## 6. (Opcional) Exportar modelo entrenado

Descarga el `.joblib` para probarlo en el microservicio local (`ml/app/`).

In [ ]:
model_path = Path("riesgo_equipo_colab_poc.joblib")
joblib.dump(pipeline, model_path)
print(f"Modelo guardado: {model_path} ({model_path.stat().st_size:,} bytes)")

try:
    from google.colab import files
    files.download(str(model_path))
except ImportError:
    print("No estás en Colab; el archivo quedó en el directorio local.")

## Checklist POC

- [ ] CSV cargado sin errores (200 filas recomendadas)
- [ ] Distribución de `nivel_riesgo` razonable (no una sola clase)
- [ ] Reglas del target ≥ 95% coincidencia
- [ ] Accuracy test ≥ 0.85 en sintético (referencia proyecto: ~0.95)
- [ ] Features más importantes alineadas con negocio (correctivos, fallas, antigüedad)

**Siguiente paso:** exportar datos reales con `python ml/scripts/build_dataset.py` y repetir este notebook con `ml/data/processed/equipos_riesgo_v1.csv`.